In [ ]:
import ipaddress
import pandas as pd
import matplotlib.pyplot as plt

# Keep consistent with other analysis notebooks
TIME_OFFSET = 10800

In [ ]:
def plot_write_packet_count_vs_nunique_trans_id_per_src_ip(
    input_csv,
    src_ip,
    center_timestamp,
    interval_seconds,
    func_col="modbus.func_code",
    trans_id_col="mbtcp.trans_id",
    write_func_codes=(5, 6, 15, 16, 22, 23),
    time_offset_seconds=TIME_OFFSET,
):
    """
    1) Plot packet_count vs nunique(mbtcp.trans_id) per src IP, considering only write-function packets.
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", func_col, trans_id_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(
        epoch[epoch.notna()] + float(time_offset_seconds),
        unit="s",
        errors="coerce",
    )

    func_num = pd.to_numeric(df[func_col], errors="coerce")
    write_codes = {int(c) for c in write_func_codes}
    df = df[func_num.isin(write_codes)].copy()

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    src_df = df[df["ip.src"].astype(str) == src_ip].copy()
    window_df = src_df[(src_df["aligned_ts"] >= start_ts) & (src_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    packet_count = window_df.groupby("second_bin").size().reindex(bins, fill_value=0)
    nunique_trans_id = window_df.groupby("second_bin")[trans_id_col].nunique().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, packet_count.values, marker="o", label="write packet_count")
    plt.plot(rel_x, nunique_trans_id.values, marker="o", label=f"write nunique({trans_id_col})")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel("count")
    plt.title(f"Write packet_count vs nunique(trans_id) for src={src_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return packet_count, nunique_trans_id


def plot_write_nunique_reference_num_per_src_ip(
    input_csv,
    src_ip,
    center_timestamp,
    interval_seconds,
    func_col="modbus.func_code",
    ref_col="modbus.reference_num",
    write_func_codes=(5, 6, 15, 16, 22, 23),
    time_offset_seconds=TIME_OFFSET,
):
    """
    2) Plot nunique(modbus.reference_num) per src IP, considering only write-function packets.
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", func_col, ref_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(
        epoch[epoch.notna()] + float(time_offset_seconds),
        unit="s",
        errors="coerce",
    )

    func_num = pd.to_numeric(df[func_col], errors="coerce")
    write_codes = {int(c) for c in write_func_codes}
    df = df[func_num.isin(write_codes)].copy()

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    src_df = df[df["ip.src"].astype(str) == src_ip].copy()
    window_df = src_df[(src_df["aligned_ts"] >= start_ts) & (src_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")[ref_col].nunique().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"write nunique({ref_col})")
    plt.title(f"Unique write reference numbers for src={src_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


def plot_write_read_ratio_per_src_ip(
    input_csv,
    src_ip,
    center_timestamp,
    interval_seconds,
    func_col="modbus.func_code",
    write_func_codes=(5, 6, 15, 16, 22, 23),
    read_func_codes=(1, 2, 3, 4),
    time_offset_seconds=TIME_OFFSET,
):
    """
    3) Plot write / (write + read) ratio per src IP in 1-second bins.

    Only packets whose function code is in write_func_codes or read_func_codes
    are used in the denominator.
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", func_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(
        epoch[epoch.notna()] + float(time_offset_seconds),
        unit="s",
        errors="coerce",
    )

    func_num = pd.to_numeric(df[func_col], errors="coerce")
    write_codes = {int(c) for c in write_func_codes}
    read_codes = {int(c) for c in read_func_codes}

    df["is_write"] = func_num.isin(write_codes).astype(int)
    df["is_read"] = func_num.isin(read_codes).astype(int)

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    src_df = df[df["ip.src"].astype(str) == src_ip].copy()
    window_df = src_df[(src_df["aligned_ts"] >= start_ts) & (src_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    write_count = window_df.groupby("second_bin")["is_write"].sum().reindex(bins, fill_value=0)
    read_count = window_df.groupby("second_bin")["is_read"].sum().reindex(bins, fill_value=0)

    denom = write_count + read_count
    ratio = (write_count / denom.where(denom != 0)).fillna(0.0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, ratio.values, marker="o", label="write/(write+read)")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.ylim(-0.05, 1.05)
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel("ratio")
    plt.title(f"Write ratio for src={src_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return ratio


def plot_write_nunique_regval_per_src_ip(
    input_csv,
    src_ip,
    center_timestamp,
    interval_seconds,
    func_col="modbus.func_code",
    regval_col="modbus.regval_uint16",
    write_func_codes=(5, 6, 15, 16, 22, 23),
    time_offset_seconds=TIME_OFFSET,
):
    """
    4) Plot nunique(modbus.regval_uint16) per src IP, considering only write-function packets.
    """
    ipaddress.ip_address(src_ip)
    df = pd.read_csv(input_csv)

    required_cols = {"ip.src", "frame.time_epoch", func_col, regval_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    epoch = pd.to_numeric(df["frame.time_epoch"], errors="coerce")
    df = df.loc[epoch.notna()].copy()
    df["aligned_ts"] = pd.to_datetime(
        epoch[epoch.notna()] + float(time_offset_seconds),
        unit="s",
        errors="coerce",
    )

    func_num = pd.to_numeric(df[func_col], errors="coerce")
    write_codes = {int(c) for c in write_func_codes}
    df = df[func_num.isin(write_codes)].copy()

    center_ts = pd.to_datetime(center_timestamp, errors="raise")
    start_ts = center_ts - pd.Timedelta(seconds=int(interval_seconds))
    end_ts = center_ts + pd.Timedelta(seconds=int(interval_seconds))

    src_df = df[df["ip.src"].astype(str) == src_ip].copy()
    window_df = src_df[(src_df["aligned_ts"] >= start_ts) & (src_df["aligned_ts"] <= end_ts)].copy()
    window_df["second_bin"] = window_df["aligned_ts"].dt.floor("s")

    center_second = center_ts.floor("s")
    bins = pd.date_range(
        start=center_second - pd.Timedelta(seconds=int(interval_seconds)),
        end=center_second + pd.Timedelta(seconds=int(interval_seconds)),
        freq="s",
    )
    rel_x = (bins - center_second).total_seconds().astype(int)

    series = window_df.groupby("second_bin")[regval_col].nunique().reindex(bins, fill_value=0)

    plt.figure(figsize=(11, 4))
    plt.plot(rel_x, series.values, marker="o")
    plt.axvline(0, color="black", linestyle="--", linewidth=1, label="Target timestamp")
    plt.xlabel("Seconds offset from target timestamp")
    plt.ylabel(f"write nunique({regval_col})")
    plt.title(f"Unique write register values for src={src_ip} around {center_ts} (+/-{interval_seconds}s)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return series


# Example usage:
# input_csv = "../train/cscada_attack_ssw.csv"
# src_ip = "185.175.0.5"
# center_timestamp = "2023-03-19 03:01:57.813"
# x = 20
#
# plot_write_packet_count_vs_nunique_trans_id_per_src_ip(input_csv, src_ip, center_timestamp, x)
# plot_write_nunique_reference_num_per_src_ip(input_csv, src_ip, center_timestamp, x)
# plot_write_read_ratio_per_src_ip(input_csv, src_ip, center_timestamp, x)
# plot_write_nunique_regval_per_src_ip(input_csv, src_ip, center_timestamp, x)